# 🛰️ Hyperspectral Image Classification — Minor Project (Test Run)

> **Settings check:** In Kaggle/Colab, ensure your **GPU accelerator is enabled** (e.g. GPU T4 x2).

Run each cell sequentially from top to bottom.

### ⚡ Cell 1 — Check GPU

In [ ]:
import torch
print("PyTorch version :", torch.__version__)
print("CUDA available  :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name        :", torch.cuda.get_device_name(0))
    print("GPU memory (GB) :", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))
else:
    print("⚠️ No GPU found — please enable GPU in settings (Accelerator -> GPU T4)")

### 📥 Cell 2 — Clone Repository

In [ ]:
!git clone https://github.com/Tanishq251/Hyperspectral-Image-Models.git
print("\n✅ Repository cloned successfully!")

### 📂 Cell 3 — Navigate to Working Directory

In [ ]:
import os
# Change directory for Kaggle (if on Colab, use /content/Hyperspectral-Image-Models)
target_dir = "/kaggle/working/Hyperspectral-Image-Models" if os.path.exists("/kaggle/working") else "Hyperspectral-Image-Models"
os.chdir(target_dir)
print("Current directory:", os.getcwd())

### ⚙️ Cell 4 — Install Compatible Packages (Locks NumPy < 2.0 & Pandas < 2.3)

> We pin `numpy<2.0` and `pandas<2.3` first to prevent binary compatibility errors.

In [ ]:
print("1. Installing compatible numpy & pandas versions...")
!pip install "numpy<2.0" "pandas<2.3" -q

print("2. Installing core framework requirements...")
!pip install -r requirements.txt -q

print("3. Installing optional packages...")
!pip install -r requirements-optional.txt -q

print("\n✅ Packages installed cleanly!")

### 📄 Cell 5 — Create Test Configuration File

In [ ]:
config = """
# ============================================================
#  TEST LIGHT CONFIG — Smoke test before Mamba/SSM experiments
# ============================================================

results:
  directory: "Results/Test_Light"
  keep_best_worst_checkpoints_only: True
  checkpoint_pruning_metric: "OA"

dataset:
  names: ["Indian_Pines"]
  run_all_datasets: False
  patch_size: 11
  stride: 1
  verbose: True

model:
  name: ["HybridSN", "SSRN", "pResNet", "DBDA", "SpectralFormer"]
  run_all_models: False
  print_summary: True
  summary_only: False
  summary_depth: 1

data_split:
  method: "samples"
  disjoint: False
  split_samples: [10, 5]
  split_samples_overrides: {}
  split_ratios: null
  random_state: 42
  seeds: [1, 2]
  print_stats: True

preprocessing:
  dim_reduction_method: pca
  num_pca_bands: 30
  maxpool_kernel: 2
  use_channel_dim: True
  band_indices: null

training:
  num_epochs: 30
  num_runs: 2
  batch_size: 32
  learning_rate: 0.001
  optimizer: "adam"
  optimizer_params: {}
  patience: 8
  checkpoint_interval: 10
  num_workers: 2

device:
  use_cuda: True
  cuda_device: 0

visualization:
  generate_maps: True
  mode: "labeled_only"
  use_spy_colors: True
  block_background: True
  show_colorbar: False
  dpi: 150

map_arrangement:
  enabled: False

score_arrangement:
  all_datasets: True
  all_models: True
  models: []
  use_mean: True
"""

with open("config/test_light.yaml", "w") as f:
    f.write(config)

print("✅ Configuration written to config/test_light.yaml")

### 🔍 Cell 6 — Verify Available Models

In [ ]:
!python main.py --list-models

### 🚀 Cell 7 — Run Training (15–30 mins)

> If training was already completed previously, skip this cell.

In [ ]:
!python main.py config/test_light.yaml

### 📊 Cell 8 — Generate LaTeX Results Table

In [ ]:
!python main.py config/test_light.yaml --arrange-scores

### 📁 Cell 9 — Check Output Results Structure

In [ ]:
import os

print("Directory tree in Results/Test_Light:\n")
for root, dirs, files in os.walk("Results/Test_Light"):
    level = root.replace("Results/Test_Light", "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    for f in files:
        print(f"{'  ' * (level+1)}{f}")

### 🖼️ Cell 10 — View Classification Maps

In [ ]:
import glob
import matplotlib.pyplot as plt
from PIL import Image

maps = glob.glob("Results/Test_Light/**/*.png", recursive=True)
print(f"Found {len(maps)} classification map(s)")

if maps:
    n = len(maps)
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 4))
    if n == 1:
        axes = [axes]
    for ax, path in zip(axes, maps):
        model_name = path.split("/")[-3]
        img = Image.open(path)
        ax.imshow(img)
        ax.set_title(model_name, fontsize=11)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

### 📋 Cell 11 — View Results Summary Table with Pandas

In [ ]:
import glob
import pandas as pd

csvs = glob.glob("Results/Test_Light/**/results_summary.csv", recursive=True)
print(f"Found {len(csvs)} results_summary file(s):\n")

all_dfs = []
for csv_path in csvs:
    model_name = csv_path.split("/")[-2]
    df = pd.read_csv(csv_path)
    df.insert(0, "Model", model_name)
    all_dfs.append(df)

if all_dfs:
    combined = pd.concat(all_dfs, ignore_index=True)
    pd.set_option("display.float_format", "{:.4f}".format)
    display(combined)
else:
    print("⚠️ No results_summary.csv found. Check if Cell 7 completed successfully.")

### 📦 Cell 12 — Zip Output for Easy Download

In [ ]:
import shutil
shutil.make_archive("/kaggle/working/HSI_Test_Results", "zip", "Results/Test_Light")
print("✅ Saved as /kaggle/working/HSI_Test_Results.zip")
print("You can download it directly from the Output files tab on the right side panel!")